In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("src")

import torch
import gc
import pandas as pd
from tqdm import tqdm

import _prompt
import _mapping
import _util
from _intervention import get_label_probability

In [3]:
_util.print_GPU_availbility()

CUDA is available: True
Available devices:
  GPU 0: NVIDIA RTX A5500
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B

In [4]:
model_type = "GPT-OSS_stepwise" # GPT-OSS or R1

if "GPT-OSS" in model_type:
    model, tokenizer = _util.load_OSS()
elif "R1" in model_type:
    model, tokenizer = _util.load_R1()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [5]:
prompt_type = "h_pre_penultimate_sum" # empty or pre_result or pre_sum
if prompt_type:
    prompt_type = "_" + prompt_type

# Load the divided prompts dataset
if 'h' in prompt_type:
    prompts = pd.read_csv(f"data/{model_type}/h_prompts{prompt_type[2:]}.csv")
else:
    prompts = pd.read_csv(f"data/{model_type}/prompts{prompt_type}.csv")
prompts["base_sum"] = prompts["base_sum"].astype('Int64')
prompts["source_sum"] = prompts["source_sum"].astype('Int64')
print(f"loaded {len(prompts)} divided prompts")

loaded 256 divided prompts


In [6]:
intervention_loc = [20] # restatement or reasoning or restatement_and_reasoning

if type(intervention_loc) == str:
    if 'h' in prompt_type:
        if model_type == "GPT-OSS_stepwise":
            intervention_ids_dict = _mapping.intervene_ids_stepwise_3_digit_h
        elif model_type == "GPT-OSS_vanilla":
            intervention_ids_dict = _mapping.intervene_ids_vanilla_2_digit_h
        elif model_type == "R1":
            intervention_ids_dict = _mapping.intervene_ids_R1_3_digit_h
        else:
            raise ValueError(f"Invalid model type: {model_type}")
    else:
        if model_type == "GPT-OSS":
            intervention_ids_dict = _mapping.intervene_ids_stepwise_3_digit
        elif model_type == "R1":
            intervention_ids_dict = _mapping.intervene_ids_R1_3_digit
        else:
            raise ValueError(f"Invalid model type: {model_type}")

    if intervention_loc == "restatement":
        intervention_ids = intervention_ids_dict["restatement"]
    elif intervention_loc == "reasoning":
        intervention_ids = intervention_ids_dict["reasoning"]
    elif intervention_loc == "restatement_and_reasoning":
        intervention_ids = intervention_ids_dict["restatement"] + intervention_ids_dict["reasoning"]
elif type(intervention_loc) == list:
    intervention_ids = intervention_loc

tok_pos_dict = _mapping.intervene_id_to_tok_pos_stepwise_3_digit_h
tok_pos_list = [tok_pos_dict[id] for id in intervention_ids]
print(intervention_ids)
print(tok_pos_list)

[20]
[235]


In [7]:
# Get header of divided prompts dataset
header = list(prompts.columns) + ['layer', 'head_num', 'faithful_diff', 'unfaithful_diff']

filepath = _util.create_csv_file(f"experiments/attention_heads/output/{model_type}/pattern_diff", f"{prompt_type[1:]}.csv", header, overwrite=False)

batch_size = 4

for i in tqdm(range(0, len(prompts), batch_size)):
    torch.cuda.empty_cache()
    gc.collect()
    batch_rows = prompts.iloc[i:i+batch_size]
    
    # Tokenize all prompts in the batch
    clean_tokens = tokenizer(batch_rows['base_prompt'].tolist(), add_special_tokens=False, return_tensors="pt", padding=True, padding_side="left").to(model.device)
    # Generate for the entire batch
    clean_output = model(
        clean_tokens.input_ids,
        attention_mask=clean_tokens.attention_mask,
        pad_token_id=tokenizer.pad_token_id,
        output_attentions=True,
    )
    # Pick out relevant attention patterns
    clean_faithful_attentions = torch.stack([layer_attention[:,:,-1,tok_pos_list[0]] for layer_attention in clean_output.attentions])
    clean_unfaithful_attentions = torch.stack([layer_attention[:,:,-1,tok_pos_list[0]-2] for layer_attention in clean_output.attentions])
    
    # Prepare batch of intervention prompts
    intervened_prompts = [_prompt.get_intervened_prompt(intervention_ids, row['base_prompt'], row['source_prompt']) for _, row in batch_rows.iterrows()]
    # Tokenize all prompts in the batch
    intervened_tokens = tokenizer(intervened_prompts, add_special_tokens=False, return_tensors="pt", padding=True, padding_side="left").to(model.device)
    # Generate for the entire batch
    intervened_output = model(
        intervened_tokens.input_ids,
        attention_mask=intervened_tokens.attention_mask,
        pad_token_id=tokenizer.pad_token_id,
        output_attentions=True,
    )
    # Pick out relevant attention patterns
    intervened_faithful_attentions = torch.stack([layer_attention[:,:,-1,tok_pos_list[0]] for layer_attention in intervened_output.attentions])
    intervened_unfaithful_attentions = torch.stack([layer_attention[:,:,-1,tok_pos_list[0]-2] for layer_attention in intervened_output.attentions])
    
    attention_faithful_diffs = intervened_faithful_attentions - clean_faithful_attentions
    attention_unfaithful_diffs = intervened_unfaithful_attentions - clean_unfaithful_attentions
    
    # Process each generated text in the batch
    for j, (_, row) in enumerate(batch_rows.iterrows()):
        for layer in range(model.config.num_hidden_layers):
            for head in range(model.config.num_attention_heads):
                _util.write_to_csv(filepath, row.to_list() + [layer, head, attention_faithful_diffs[layer, j, head].item(), attention_unfaithful_diffs[layer, j, head].item()])

    del clean_tokens, clean_output, intervened_tokens, intervened_output, clean_faithful_attentions, clean_unfaithful_attentions, intervened_faithful_attentions, intervened_unfaithful_attentions, attention_faithful_diffs, attention_unfaithful_diffs
    torch.cuda.empty_cache()
    gc.collect()


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 64/64 [16:16<00:00, 15.27s/it]
